# Alunos
- **Matheus Peixoto Ribeiro Vieira - 22.1.4104**
- **Pedro Henrique Rabelo Leão de Oliveira - 22.1.4022**

# Mineração de Regras de Associação

In [37]:
import pandas as pd

import json
with open('data/BRA-modified.json', 'r') as f:
    data = json.load(f)

data = {id: name for name, id in zip(data.keys(), data.values())}
data

id_to_name = lambda id: data[id]

## Discretizando os valores numéricos

In [38]:
def freedman_diaconis_rule(data):
    iqr = data.quantile(0.75) - data.quantile(0.25)
    data = data.to_numpy()
    num_bins = int((data.max()-data.min()) / (2*iqr*len(data)**(-1/3)))
    return num_bins

In [39]:
# dataset sem o pre-processamento feito no arquivo 'pre_processing.ipynb'
original_dataset = pd.read_csv('data/BRA.csv')
original_dataset = original_dataset.drop(1891)

In [40]:
def discretiza_odds(valor_odd):
    """ 
    Discretiza os valores das odds, classificando em: 
     - 0 (time favorito da partida) quando a odd é menor que 2; 
     - 1 (equilibrado) quando a odd está no intervalo [2, 3[;
     - 2 (time improvável de vencer) quando a odd é maior ou igual a 3.
    """
    if valor_odd < 2:
        return 0
    elif valor_odd < 3:
        return 1
    else:
        return 2

df = pd.read_csv('BRA-pre-processed.csv')
categorical_data = ["Home", "Away", "Res_D", "Res_H", "WLF_H", "DLF_H", "LLF_H", "WLF_A", "DLF_A", "LLF_A"]
numerical_columns = list(set(df.columns.to_list()) - set(categorical_data))

for col in numerical_columns:
    if col in ["PSCH", "PSCD", "PSCA", "MaxCH", "MaxCD", "MaxCA", "AvgCA", "AvgCH", "AvgCD"]:
        df[col] = original_dataset[col].apply(discretiza_odds)

    else:
        num_bins = freedman_diaconis_rule(df[col])
        value_bins = pd.cut(df[col], num_bins, labels=range(0, num_bins))
        df[col] = value_bins


In [41]:
df

,Home,Away,PSCH,PSCD,PSCA,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,...,GA_A,GD_A,WLF_H,DLF_H,LLF_H,WLF_A,DLF_A,LLF_A,Res_D,Res_H
0,27,30,0.0,2.0,2.0,0.0,2.0,2.0,0.0,2.0,...,0,47,0.0,0.0,0.0,0.0,0.0,0.0,1,0
1,34,17,1.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0,...,0,47,0.0,0.0,0.0,0.0,0.0,0.0,1,0
2,16,26,0.0,2.0,2.0,0.0,2.0,2.0,0.0,2.0,...,0,47,0.0,0.0,0.0,0.0,0.0,0.0,0,1
3,6,33,1.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,...,0,47,0.0,0.0,0.0,0.0,0.0,0.0,0,1
4,11,18,0.0,2.0,2.0,0.0,2.0,2.0,0.0,2.0,...,0,47,0.0,0.0,0.0,0.0,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5050,36,14,1.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,...,2,55,0.2,0.2,0.6,0.8,0.2,0.0,1,0
5051,19,32,2.0,2.0,1.0,2.0,2.0,1.0,2.0,2.0,...,4,44,0.2,0.2,0.6,0.2,0.2,0.6,0,0
5052,21,11,1.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,...,5,46,0.6,0.2,0.2,0.4,0.4,0.2,1,0
5053,3,22,1.0,1.0,2.0,1.0,2.0,2.0,1.0,1.0,...,5,44,0.6,0.4,0.0,0.0,0.4,0.6,0,1


## Regras de Associação

In [42]:
transactions = []

for index_row, row in df.iterrows():
    res = ('H' if (row['Res_H'] == 1) else 
           'D' if (row['Res_D'] == 1) else 
           'A')

    num_goals = original_dataset.iloc[index_row]['HG'] + original_dataset.iloc[index_row]['AG']
    total_goals = "Mais_2.5_Gols" if (num_goals >= 3) else "Menos_2.5_Gols"

    odd_media_casa = ('Odd_Casa_Baixa' if (row['AvgCH'] == 0) else 
                      'Odd_Casa_Média' if (row['AvgCH'] == 1) else 
                      'Odd_Casa_Alta')
    
    odd_media_empate = ('Odd_Empate_Baixa' if (row['AvgCD'] == 0) else 
                        'Odd_Empate_Média' if (row['AvgCD'] == 1) else 
                        'Odd_Empate_Alta')
    
    odd_media_visitante = ('Odd_Visitante_Baixa' if (row['AvgCA'] == 0) else 
                           'Odd_Visitante_Média' if (row['AvgCA'] == 1) else 
                           'Odd_Visitante_Alta')

    itens = [
        f"Casa_{int(row['Home'])}",
        f"Visitante_{int(row['Away'])}",
        total_goals, # total de gols na partida
        f"Vitórias_Casa_{row['WLF_H']}",     # Vítorias do time da casa nas 5 partidas anteriores
        f"Vitórias_Visitante_{row['WLF_A']}", # Vítorias do time visitante nas 5 partidas anteriores
        f"Resultado_{res}",
        odd_media_casa,
        odd_media_empate,
        odd_media_visitante
    ]

    transactions.append(itens)

for t in transactions[:5]:
    print(t)

['Casa_27', 'Visitante_30', 'Menos_2.5_Gols', 'Vitórias_Casa_0.0', 'Vitórias_Visitante_0.0', 'Resultado_D', 'Odd_Casa_Baixa', 'Odd_Empate_Alta', 'Odd_Visitante_Alta']
['Casa_34', 'Visitante_17', 'Menos_2.5_Gols', 'Vitórias_Casa_0.0', 'Vitórias_Visitante_0.0', 'Resultado_D', 'Odd_Casa_Média', 'Odd_Empate_Alta', 'Odd_Visitante_Média']
['Casa_16', 'Visitante_26', 'Mais_2.5_Gols', 'Vitórias_Casa_0.0', 'Vitórias_Visitante_0.0', 'Resultado_H', 'Odd_Casa_Baixa', 'Odd_Empate_Alta', 'Odd_Visitante_Alta']
['Casa_6', 'Visitante_33', 'Mais_2.5_Gols', 'Vitórias_Casa_0.0', 'Vitórias_Visitante_0.0', 'Resultado_H', 'Odd_Casa_Média', 'Odd_Empate_Alta', 'Odd_Visitante_Média']
['Casa_11', 'Visitante_18', 'Menos_2.5_Gols', 'Vitórias_Casa_0.0', 'Vitórias_Visitante_0.0', 'Resultado_A', 'Odd_Casa_Baixa', 'Odd_Empate_Alta', 'Odd_Visitante_Alta']


### Executando o apriori

A biblioteca mlxtend.frequent_patterns necessita que todos os dados estejam em formato one-hot encoded. Todavia trabalhamos com valores númericos contínuos. Dessa forma, transformar a discretização de todos os atributos em one-hot elevaria muito o custo de memória. Por isso o seu uso não será escolhido e utilizaremos o apyori.

In [43]:
from apyori import apriori

In [44]:
association_rules = apriori(transactions, min_support=0.2, min_confidence=0.6, min_lift=1.2) 
association_results = list(association_rules)

for r in association_results:
    itens = [x for x in r.items]
    print(f"Regra: {itens}")
    print(f"Suporte: {r.support:.3f}")
    for o in r.ordered_statistics:
        print(f"  {list(o.items_base)} => {list(o.items_add)}")
        print(f"  Confiança: {o.confidence:.3f}")
        print(f"  Lift: {o.lift:.3f}")
    print("-" * 50)

Regra: ['Menos_2.5_Gols', 'Resultado_D']
Suporte: 0.216
  ['Resultado_D'] => ['Menos_2.5_Gols']
  Confiança: 0.802
  Lift: 1.409
--------------------------------------------------
Regra: ['Odd_Visitante_Alta', 'Odd_Casa_Baixa']
Suporte: 0.475
  ['Odd_Casa_Baixa'] => ['Odd_Visitante_Alta']
  Confiança: 1.000
  Lift: 1.414
  ['Odd_Visitante_Alta'] => ['Odd_Casa_Baixa']
  Confiança: 0.671
  Lift: 1.414
--------------------------------------------------
Regra: ['Odd_Visitante_Alta', 'Mais_2.5_Gols', 'Odd_Casa_Baixa']
Suporte: 0.210
  ['Mais_2.5_Gols', 'Odd_Casa_Baixa'] => ['Odd_Visitante_Alta']
  Confiança: 1.000
  Lift: 1.414
  ['Odd_Visitante_Alta', 'Mais_2.5_Gols'] => ['Odd_Casa_Baixa']
  Confiança: 0.687
  Lift: 1.447
--------------------------------------------------
Regra: ['Menos_2.5_Gols', 'Odd_Visitante_Alta', 'Odd_Casa_Baixa']
Suporte: 0.264
  ['Menos_2.5_Gols', 'Odd_Casa_Baixa'] => ['Odd_Visitante_Alta']
  Confiança: 1.000
  Lift: 1.414
  ['Menos_2.5_Gols', 'Odd_Visitante_Alta']

(Suporte Mínimo 0.3, confiança mínima 0.2 e lift mínimo 0.8)
- O item "Odd_Visitante_Baixa" não atendeu ao suporte mínimo, o que confirma a questão do favoritismo presente para os times que jogam em casa no brasileirão vista nas análises feitas por nós na primeira fase do trabalho. 
- Foi possível observar também a presença de odds altas para empates nas partidas, presente em 91,8% das partidas. 
- Além disso, 57% das partidas tiveram 2 ou menos gols, o que confirma o quão equilibrado o campeonato brasileiro é. 
- Uma regra encontrada foi o time da casa vencendo 48,4% das partidas, valor esses encontrado por nós nas análises feitas durante a primeira fase do trabalho e confirmando mais uma vez o favoritismo de quem joga em casa, ao lado de sua torcida.
- Em relação a regras com antecedentes, foi possível observar que toda vez que a odd média para o time da casa era baixa a odd média para o time visitante era alta, tendo assim uma confiança de 100%, o que é esperado no momento em que há o favoritismo para o time da casa. 
- Por fim, em 51,1% das vezes que temos uma odd média alta para o visitante, temos o time da casa vencendo a partida.

(Suporte Mínimo 0.2, confiança mínima de 0.6 e lift mínimo de 1.2)
- Foi possível observar que quando temos um empate, em 80,2% das vezes, acontecem 2 ou menos gols na partida, e o lift de 1,409 traz uma relação forte para o antecedente e o consequente. Isso condiz com o equilíbrio que se espera de uma partida que termina empatada, onde, nesses casos, acontecem menos gols.
- Aqui, foi possível confirmar também a questão da presença da odd média alta para o visitante em 100% das vezes que temos a odd média baixa para o time da casa.
- Além disso, no momento em que temos o resultado da partida com o time da casa vencendo e a odd média para esse time baixa, em 100% das vezes as odds médias para o visitante e para empate são altas, confirmando o favoritismo do time da casa.
- É possível perceber que uma das regras é composta pela odd média baixa para o time da casa, time da casa vencendo e odds médias altas para o time visitante e para o empate, o que indica que esses atributos estão relacionados normalmente.

(Suporte Mínimo 0.3, confiança mínima de 0.3 e lift mínimo de 0.9)
- Foi possível observar que, sempre que temos a presença da odd média baixa para o time da casa, também temos a presença das odds médias altas para o empate e para o visitante, com confiança de 100%, o que reforça o favoritismo do time da casa e a baixa expectativa de equilíbrio no confronto.
- A recíproca também ocorre, onde odds altas para o visitante ou para o empate implicam na odd baixa para o time da casa, com confianças de 67,1% e 51,7%, respectivamente, mostrando consistência na forma como as odds refletem o desequilíbrio entre as equipes.
- Em relação ao número de gols, foi identificado que quando ocorrem mais de 2.5 gols na partida, é comum a presença de odds altas para o empate e para o visitante, com confianças de 91,4% e 71,2%, respectivamente. Já quando ocorrem menos de 2.5 gols, há uma regra frequente envolvendo também odds altas para empate e visitante, com confiança de 67,3%.
- Em relação aos resultados, observou-se que os desfechos desfavoráveis ao time da casa não são frequentes com os hiperparâmetros utilizados, o que reforça o padrão de favoritismo do mandante.
- Também foi possível verificar que, quando temos odds altas tanto para o empate quanto para o visitante, há elevada confiança para a vitória do time da casa, com valores de confiança que variam, sendo o mínimo de 34,5%, mesmo sem antecedentes adicionais, reforçando a força do mandante nas partidas.

# Padrões sequenciais

In [78]:
"""
    Sequência de resultados de cada um dos times ao longo dos anos
    Itemset é formado por
        - Resultado desse time na partida: W (Win), D (Draw), L (Lose)
        - Quantos gols fez, mais ou menos do que 2.5
        - Quantos gols levou, mais ou menos do que 2.5
"""
sequencia_de_resultados = dict()

"""
    Sequência de confrontos entre dois times levando em consideração o time mandante
    Time A X Time B é uma sequência, enquanto Time B X Time A é outra
    Itemset formado por:
        - Resultado: H (Home), D (Draw), A (Away)
        - Quantos gols tiveram na partida, mais ou menos que 2.5
        - Categorização das Odds de vitória para casa, empate e visitante em Baixa, Média ou Alta
"""
sequencias_de_confrontos_casa_visitante = dict()

"""
    Sequência de confrontos entre dois times
    Time A X Time B é a mesma sequência de Time B X Time A
    Time A é sempre o de menor ID
    Itemset formado por:
        - Resultado: Nome do time A, Empate, Nome do time visitante
        - Quantos gols tiveram na partida, mais ou menos que 2.5
        - Categorização das Odds de vitória para Time A, empate e Time B em Baixa, Média ou Alta
"""
sequencias_de_confrontos_independente_mandante = dict()

for index_row, row in df.iterrows():
    # Sequencia de resultados em confrontos entre dois times
    home, away = row["Home"], row["Away"]

    res = ('H' if (row['Res_H'] == 1) else 
           'D' if (row['Res_D'] == 1) else 
           'A')
    
    num_goals = original_dataset.iloc[index_row]['HG'] + original_dataset.iloc[index_row]['AG']
    total_goals = "Mais_2.5_Gols" if (num_goals >= 3) else "Menos_2.5_Gols"

    odd_media_casa = ('Odd_Casa_Baixa' if (row['AvgCH'] == 0) else 
                      'Odd_Casa_Média' if (row['AvgCH'] == 1) else 
                      'Odd_Casa_Alta')
    
    odd_media_empate = ('Odd_Empate_Baixa' if (row['AvgCD'] == 0) else 
                        'Odd_Empate_Média' if (row['AvgCD'] == 1) else 
                        'Odd_Empate_Alta')
    
    odd_media_visitante = ('Odd_Visitante_Baixa' if (row['AvgCA'] == 0) else 
                           'Odd_Visitante_Média' if (row['AvgCA'] == 1) else 
                           'Odd_Visitante_Alta')
    
    partida = [res, total_goals, odd_media_casa, odd_media_empate, odd_media_visitante]

    if (home, away) in sequencias_de_confrontos_casa_visitante:
        sequencias_de_confrontos_casa_visitante[(home, away)].append(partida)
    else:
        sequencias_de_confrontos_casa_visitante[(home, away)] = [partida]

    # Sequencia de resultados de um time especifico
    if res == "H": 
        h="W"
        a="L"
    elif res == "D": 
        h=a="D"
    else: 
        h="L"
        a="W"
    
    hg = "Mais_2.5_Gols" if (original_dataset.iloc[index_row]['HG'] >= 3) else "Menos_2.5_Gols"
    ag = "Mais_2.5_Gols" if (original_dataset.iloc[index_row]['AG'] >= 3) else "Menos_2.5_Gols"

    if home in sequencia_de_resultados:
        sequencia_de_resultados[home].append([h, f"Fez_{hg}", f"Levou_{ag}"])
    else:
        sequencia_de_resultados[home] = [[h, f"Fez_{hg}", f"Levou_{ag}"]]

    if away in sequencia_de_resultados:
        sequencia_de_resultados[away].append([a, f"Fez_{ag}", f"Levou_{hg}"])
    else:
        sequencia_de_resultados[away] = [[a, f"Fez_{ag}", f"Levou_{hg}"]]

    # sequencias_de_confrontos_independente_mandante
    team_a = id_to_name(int(min(home, away)))
    team_b = id_to_name(int(max(home, away)))

    res = (id_to_name(home) if (row['Res_H'] == 1) else 
           'Empate' if (row['Res_D'] == 1) else 
           id_to_name(away))

    odd_media_casa = odd_media_casa.replace("Casa", id_to_name(home))
    odd_media_visitante = odd_media_visitante.replace("Visitante", id_to_name(away))
    odds = sorted([odd_media_casa, odd_media_visitante])
    partida = [res, total_goals, odds[0], odd_media_empate, odds[1]]

    if (team_a, team_b) in sequencias_de_confrontos_independente_mandante:
        sequencias_de_confrontos_independente_mandante[(team_a, team_b)].append(partida)
    else:
        sequencias_de_confrontos_independente_mandante[(team_a, team_b)] = [partida]

In [46]:
print(sequencia_de_resultados[11])
print(sequencias_de_confrontos_casa_visitante[(3, 14)])
print(sequencias_de_confrontos_independente_mandante[('Atletico-MG', 'Cruzeiro')])

[['L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['L', 'Fez_Menos_2.5_Gols', 'Levou_Mais_2.5_Gols'], ['W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['W', 'Fez_Mais_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], ['L', 'Fez_Menos_2.5_Gols', 'Levou_Mais_2.5_Gols'], ['L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gol

## Sequência de resultados de um único time ao longo das temporadas

In [211]:
from prefixspan import PrefixSpan

for k in sequencia_de_resultados:
    # Iniciar a mineração
    ps = PrefixSpan(sequencia_de_resultados[k])

    # Padrões com suporte mínimo de 40%
    minsup = int(0.4 * len(sequencia_de_resultados[k]))
    padroes = ps.frequent(minsup=minsup)

    # Exibir
    if len(padroes) > 0:
        print(id_to_name(int(k)))

        for suporte, padrao in padroes:
            print(f"Suporte: {suporte}  Padrão: {padrao}")

        print("="*50)

Palmeiras
Suporte: 381  Padrão: ['Fez_Menos_2.5_Gols']
Suporte: 350  Padrão: ['Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols']
Suporte: 431  Padrão: ['Levou_Menos_2.5_Gols']
Suporte: 229  Padrão: ['W']
Suporte: 226  Padrão: ['W', 'Levou_Menos_2.5_Gols']
Portuguesa
Suporte: 66  Padrão: ['Fez_Menos_2.5_Gols']
Suporte: 58  Padrão: ['Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols']
Suporte: 68  Padrão: ['Levou_Menos_2.5_Gols']
Sport Recife
Suporte: 288  Padrão: ['Fez_Menos_2.5_Gols']
Suporte: 244  Padrão: ['Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols']
Suporte: 264  Padrão: ['Levou_Menos_2.5_Gols']
Suporte: 138  Padrão: ['L']
Suporte: 135  Padrão: ['L', 'Fez_Menos_2.5_Gols']
Flamengo RJ
Suporte: 416  Padrão: ['Fez_Menos_2.5_Gols']
Suporte: 375  Padrão: ['Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols']
Suporte: 459  Padrão: ['Levou_Menos_2.5_Gols']
Suporte: 243  Padrão: ['W']
Suporte: 241  Padrão: ['W', 'Levou_Menos_2.5_Gols']
Figueirense
Suporte: 138  Padrão: ['Fez_Menos_2.5_Gols']
Suporte: 112  Padr

Com base nesses resultados obtidos pela execução do PrefixSpan é possível realizar uma análise ampla sobre o comportamento individual dos clubes ao longo de múltiplas temporadas do brasileirão. Algumas análises possíveis de serem feitas são:
- A maior parte das equipes, tanto grandes quanto pequenas, tem como padrão frequente a sequência Fez_Menos_2.5_Gols e Levou_Menos_2.5_Gols. Isso aparece em praticamente todos os clubes, com suporte elevado, reforçando a característica de equilíbrio e placares apertados do campeonato, onde as partidas costumam ser decididas por poucos gols.
- Times com alto suporte para vitórias, como Palmeiras, Flamengo, Atlético-MG, Internacional, Corinthians, São Paulo e Grêmio, geralmente também apresentam a regra ['W', 'Levou_Menos_2.5_Gols'] com suporte quase igual ao da própria vitória, o que indica que a maioria das vitórias desses clubes ocorre com boa atuação defensiva.
- Clubes que tradicionalmente têm campanhas mais fracas na elite, como Cuiabá, Chapecoense, América-MG, Paraná, Goias, Vitória e Avai, compartilham um padrão comum de possuirem regras frequentes como ['L'], ['L', 'Fez_Menos_2.5_Gols'] e ['Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols'], que indicam que suas derrotas vêm acompanhadas de baixo desempenho ofensivo, mesmo que não sofram goleadas.
- Poucos clubes apresentam o empate como padrão recorrente.

## Sequência de confrontos entre dois times sem verificar o mandante

In [52]:
# Verificando o maior número de conflitos
maior = 0
menor = 99
for val in sequencias_de_confrontos_independente_mandante.values():
    if len(val) > maior:
        maior=len(val)
    elif len(val) < menor:
        menor=len(val)
menor, maior

(1, 27)

In [ ]:
for k in sequencias_de_confrontos_independente_mandante:
    
    ps = PrefixSpan(sequencias_de_confrontos_independente_mandante[k])

    # Os times com mais confrontos possuem 27. O suporte mínimo será 1/3 desse valor
    padroes = ps.frequent(minsup=9)

    # Em uma analise sem o filtro, muitos resultados aparecem somente com o Odd_Empate_Alta
    so_empate_alto = len(padroes) == 1 and padroes[0][1] == ['Odd_Empate_Alta']
    if len(padroes) > 0 and not so_empate_alto:
        print(k)
        # Exibir
        for suporte, padrao in padroes:
            print(f"Suporte: {suporte}  Padrão: {padrao}")

        print("="*50)

('Flamengo RJ', 'Sport Recife')
Suporte: 11  Padrão: ['Menos_2.5_Gols']
Suporte: 10  Padrão: ['Menos_2.5_Gols', 'Odd_Empate_Alta']
Suporte: 15  Padrão: ['Odd_Empate_Alta']
Suporte: 9  Padrão: ['Odd_Empate_Alta', 'Odd_Sport Recife_Alta']
Suporte: 10  Padrão: ['Odd_Sport Recife_Alta']
Suporte: 9  Padrão: ['Flamengo RJ']
('Botafogo RJ', 'Sao Paulo')
Suporte: 13  Padrão: ['Mais_2.5_Gols']
Suporte: 13  Padrão: ['Mais_2.5_Gols', 'Odd_Empate_Alta']
Suporte: 23  Padrão: ['Odd_Empate_Alta']
Suporte: 10  Padrão: ['Odd_Empate_Alta', 'Odd_Sao Paulo_Baixa']
Suporte: 10  Padrão: ['Odd_Empate_Alta', 'Odd_Sao Paulo_Alta']
Suporte: 9  Padrão: ['Sao Paulo']
Suporte: 9  Padrão: ['Sao Paulo', 'Odd_Empate_Alta']
Suporte: 11  Padrão: ['Odd_Botafogo RJ_Alta']
Suporte: 11  Padrão: ['Odd_Botafogo RJ_Alta', 'Odd_Empate_Alta']
Suporte: 10  Padrão: ['Odd_Botafogo RJ_Alta', 'Odd_Empate_Alta', 'Odd_Sao Paulo_Baixa']
Suporte: 10  Padrão: ['Odd_Botafogo RJ_Alta', 'Odd_Sao Paulo_Baixa']
Suporte: 10  Padrão: ['Odd_Sao 

- A Odd_Empate_Alta está presente em quase todas as sequências, com excessão dos confrontos entre Ceara e São Paulo, evidenciando o fato de que, no brasileirão, as chances de ocorrerm um empate são mais baixas do que as de um time sair vitorioso. Todavia, o padrão desse jogo é ocorrer um empate

- Muitos dos jogos possuem menos de 2.5 gols e apresentam odds altas para empates. Os resultados podem ter sido tanto um 2x0 quanto 1x1


- No confronto entre Cruzeiro e Goias, o Cruzeiro tem uma maior sequência de vitorias e uma maior sequência de vitorias com a odd para empate alta.

- O Vitória costuma perder mais para São Paulo, Santos e Palmeiras

- Cruzeiro e Chapecoense possuem um padrão de, além de uma Odd alta para empate, uma odd média para o cruzeiro, mostrando que o time mineiro pode ter mais dificuldades em confrontos 

- Flamengo e Fluminense é o confronto com a maior quantidade de gols. Apresenta mais de 2.5 gols com suporte de 16

- Em geral, os confrontos do Atlético MG e os confrontos do Flamengo possuem mais frequentemente mais de 2.5 gols por partida


## Sequência de confronto entre dois times verificando o mandante

In [ ]:
for k in sequencias_de_confrontos_casa_visitante:
    # Iniciar a mineração
    ps = PrefixSpan(sequencias_de_confrontos_casa_visitante[k])

    # Padrões com suporte mínimo de 80%
    minsup = max(int(0.8 * len(sequencias_de_confrontos_casa_visitante[k])), 6)
    padroes = ps.frequent(minsup=minsup)

    if len(padroes) > 0:
        print(f'{id_to_name(int(k[0]))} X {id_to_name(int(k[1]))}')

        # Exibir
        for suporte, padrao in padroes:
            print(f"Suporte: {suporte}  Padrão: {padrao}")

        print("="*50)

Sport Recife X Flamengo RJ
Suporte: 8  Padrão: ['Odd_Empate_Alta']
Botafogo RJ X Sao Paulo
Suporte: 12  Padrão: ['Odd_Empate_Alta']
Suporte: 10  Padrão: ['Odd_Empate_Alta', 'Odd_Visitante_Alta']
Suporte: 10  Padrão: ['Odd_Visitante_Alta']
Corinthians X Fluminense
Suporte: 13  Padrão: ['Odd_Empate_Alta']
Suporte: 12  Padrão: ['Odd_Empate_Alta', 'Odd_Visitante_Alta']
Suporte: 12  Padrão: ['Odd_Visitante_Alta']
Internacional X Coritiba
Suporte: 8  Padrão: ['Odd_Empate_Alta']
Suporte: 7  Padrão: ['Odd_Empate_Alta', 'Odd_Visitante_Alta']
Suporte: 7  Padrão: ['Odd_Visitante_Alta']
Bahia X Santos
Suporte: 9  Padrão: ['Odd_Empate_Alta']
Suporte: 8  Padrão: ['Odd_Empate_Alta', 'Odd_Visitante_Alta']
Suporte: 8  Padrão: ['Odd_Visitante_Alta']
Vasco X Gremio
Suporte: 7  Padrão: ['Odd_Empate_Alta']
Suporte: 7  Padrão: ['Odd_Visitante_Alta']
Flamengo RJ X Internacional
Suporte: 13  Padrão: ['Odd_Empate_Alta']
Suporte: 12  Padrão: ['Odd_Empate_Alta', 'Odd_Visitante_Alta']
Suporte: 12  Padrão: ['Odd_V